# 实验6.3 昇腾香橙派的图片和视频YOLO目标检测实验

> 昇腾香橙派 AIPro · Ascend 310B4 NPU · YOLOv8 目标检测 · DVPP 硬件解码 + AIPP 预处理加速 + USB 摄像头实时检测 + HDMI 显示

本实验在 **昇腾香橙派 AIPro 开发板** 上运行，围绕 **YOLOv8 目标检测模型** 的端侧部署展开。实验重点实践昇腾独有的 **DVPP** 硬件视频解码与 **AIPP** 硬件预处理加速能力，并在此基础上实现 **图片目标检测**、**视频目标检测** 和 **USB 摄像头实时目标检测 + HDMI 显示** 三个完整案例。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>开发板</strong></td>
<td style="text-align: left;">昇腾香橙派 AIPro</td>
</tr>
<tr>
<td style="text-align: left;"><strong>NPU 芯片</strong></td>
<td style="text-align: left;">昇腾 310B4</td>
</tr>
<tr>
<td style="text-align: left;"><strong>操作系统</strong></td>
<td style="text-align: left;">Ubuntu 22.04 (aarch64)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN Toolkit · ATC · AscendCL · OpenCV</td>
</tr>
<tr>
<td style="text-align: left;"><strong>检测模型</strong></td>
<td style="text-align: left;">YOLOv8n（目标检测）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>输入方式</strong></td>
<td style="text-align: left;">图片文件 / 视频文件 / USB 摄像头</td>
</tr>
<tr>
<td style="text-align: left;"><strong>输出方式</strong></td>
<td style="text-align: left;">文件保存 / HDMI 实时显示</td>
</tr>
</table>

> **注意**：本 Notebook 中的相关程序运行在昇腾香橙派开发板上，此处仅说明主要的运行过程和代码结构，**不需要在线运行**。全部可执行代码存放在 `code/` 目录下，在香橙派终端中执行。

---

### 视频素材准备

`images/dog.mp4` 为本实验所用测试视频，体积较大，**不再随仓库分发**。下方单元格会在文件缺失时自动从在线地址下载到 `images/dog.mp4`；若已手动放置则跳过。


In [ ]:
import os, urllib.request

def _ensure_dog_video(path='images/dog.mp4', url='https://www.qmpan.com/f/6pLXiD/dog.mp4'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        return
    print(f'[INFO] {path} 不存在，正在从在线地址下载...')
    urllib.request.urlretrieve(url, path)
    print(f'[OK] 已下载到 {path}')

_ensure_dog_video()


## 1. 实验概述与目标

### 1.1 实验背景

昇腾香橙派 AIPro 是一款搭载昇腾 310B4 NPU 的边缘计算开发板，具备 HDMI 显示输出和 USB 接口，适合在端侧部署视觉 AI 应用。在端侧设备上部署视觉模型，面临三大挑战：

1. **算力有限**：端侧 NPU 算力远小于服务器，必须极致优化
2. **预处理开销大**：JPEG 解码、缩放、归一化等预处理在 CPU 上耗时占比高
3. **带宽受限**：Host→Device 数据搬运是瓶颈

昇腾通过 **DVPP + AIPP** 两大硬件加速能力解决这些问题：

- **DVPP**：把 JPEG 解码和图像缩放从 CPU 移到 NPU 专用硬件
- **AIPP**：把归一化和通道转换从 CPU 移到 NPU 专用硬件，并将输入数据从 float32 降为 uint8

### 1.2 实验目标

- **知识目标**：理解 DVPP 硬件视频解码加速原理；理解 AIPP 预处理卸载机制；掌握 YOLO 模型在昇腾香橙派上的端侧部署全流程
- **能力目标**：能够在香橙派上完成 YOLOv8 模型的 ATC 转换与 OM 推理；能够对比 DVPP/OpenCV 预处理与 AIPP/无 AIPP 推理路径的性能差异；能够实现 USB 摄像头实时检测并通过 HDMI 显示
- **素养目标**：形成"端侧优化→硬件加速→实时部署"的工程意识，理解边缘 AI 的全链路优化方法

### 1.3 实验内容

本实验包含三个案例：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">案例</th>
<th style="text-align: left;">输入</th>
<th style="text-align: left;">处理方式</th>
<th style="text-align: left;">输出</th>
<th style="text-align: left;">代码位置</th>
</tr>
<tr>
<td style="text-align: left;">案例一：图片检测</td>
<td style="text-align: left;"><code>images/</code> 下 4 张图片</td>
<td style="text-align: left;">DVPP+AIPP / OpenCV</td>
<td style="text-align: left;"><code>output/</code> 检测结果图</td>
<td style="text-align: left;"><code>code/yolo_image_detect.py</code></td>
</tr>
<tr>
<td style="text-align: left;">案例二：视频检测</td>
<td style="text-align: left;"><code>images/dog.mp4</code></td>
<td style="text-align: left;">DVPP+AIPP / OpenCV</td>
<td style="text-align: left;"><code>output/</code> 检测结果视频</td>
<td style="text-align: left;"><code>code/yolo_video_detect.py</code></td>
</tr>
<tr>
<td style="text-align: left;">案例三：USB摄像头实时检测</td>
<td style="text-align: left;">USB 摄像头视频流</td>
<td style="text-align: left;">DVPP+AIPP</td>
<td style="text-align: left;">HDMI 显示器实时显示</td>
<td style="text-align: left;"><code>code/yolo_usb_camera_detect.py</code></td>
</tr>
</table>

## 2. 实验环境：昇腾香橙派 AIPro

本实验在 **昇腾香橙派 AIPro** 开发板上运行。香橙派 AIPro 搭载昇腾 310B4 NPU 芯片，具备 8 TOPS @INT8 AI 算力，适合端侧视觉 AI 部署。

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">规格</th><th style="text-align: left;">参数</th></tr>
<tr><td style="text-align: left;">开发板</td><td style="text-align: left;">昇腾香橙派 AIPro</td></tr>
<tr><td style="text-align: left;">NPU 芯片</td><td style="text-align: left;">昇腾 310B4（8 TOPS @INT8）</td></tr>
<tr><td style="text-align: left;">CPU</td><td style="text-align: left;">4 核 ARM Cortex-A55 @ 1.0 GHz</td></tr>
<tr><td style="text-align: left;">内存</td><td style="text-align: left;">4 GB / 8 GB LPDDR4X</td></tr>
<tr><td style="text-align: left;">操作系统</td><td style="text-align: left;">Ubuntu 22.04 LTS (aarch64)</td></tr>
<tr><td style="text-align: left;">CANN 版本</td><td style="text-align: left;">CANN Toolkit (Ascend310B4)</td></tr>
<tr><td style="text-align: left;">Python</td><td style="text-align: left;">3.9.x / 3.10.x</td></tr>
<tr><td style="text-align: left;">视频接口</td><td style="text-align: left;">HDMI 输出（连接显示器）</td></tr>
<tr><td style="text-align: left;">USB 接口</td><td style="text-align: left;">USB 3.0 × 2（连接 USB 摄像头）</td></tr>
</table>

上表列出了香橙派 AIPro 的关键规格。**NPU** 是昇腾 310B4，AI 算力 8 TOPS @INT8，适合运行 YOLOv8n 等轻量级模型推理。**CPU** 为 4 核 ARM Cortex-A55，算力有限，因此将预处理卸载到 NPU 硬件尤为重要。**HDMI 接口** 用于连接显示器实时显示检测结果。**USB 接口** 用于连接 USB 摄像头采集视频流。与云平台不同，香橙派是 Host-Device 一体架构（NPU 和 CPU 共享内存），但仍需通过 `acl.rt.memcpy` 进行 H2D/D2H 数据搬运。

> **注意**：本 Notebook 中的程序运行在香橙派开发板上，此处仅说明运行过程，不需要在线运行。全部代码在 `code/` 目录下。

## 3. DVPP 硬件视频解码详解

### 3.1 DVPP 是什么

DVPP（Digital Vision Pre-Processing）是昇腾 NPU 内部的专用视频处理引擎，独立于 AI Core，专门处理图像/视频的编解码与几何变换。

<img src="../../images/dvpp_flow.png" alt="DVPP流水线" style="display: block; margin-left: 0;" />

### 3.2 DVPP 支持的操作

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">操作</th><th style="text-align: left;">AscendCL API</th><th style="text-align: left;">说明</th></tr>
<tr><td style="text-align: left;">JPEG 解码</td><td style="text-align: left;">acl.media.dvpp_jpeg_decode_async</td><td style="text-align: left;">JPEG → YUV420SP</td></tr>
<tr><td style="text-align: left;">VPC 缩放</td><td style="text-align: left;">acl.media.dvpp_vpc_resize_async</td><td style="text-align: left;">硬件缩放 + 裁剪</td></tr>
<tr><td style="text-align: left;">JPEG 编码</td><td style="text-align: left;">acl.media.dvpp_jpeg_encode_async</td><td style="text-align: left;">YUV420SP → JPEG</td></tr>
<tr><td style="text-align: left;">内存分配</td><td style="text-align: left;">acl.media.dvpp_malloc</td><td style="text-align: left;">DVPP 专用设备内存</td></tr>
</table>

上表列出了 DVPP 支持的四类操作。**JPEG 解码**将压缩的 JPEG 数据解码为 YUV420SP（NV12）格式，输出直接存放在 Device 内存。**VPC 缩放**不仅支持缩放，还支持裁剪、填充等几何变换。**JPEG 编码**将 YUV420SP 编码为 JPEG，常用于结果保存。**内存分配**使用 `dvpp_malloc` 而非普通 `rt.malloc`，因为 DVPP 要求内存地址按特定方式对齐。所有 DVPP 操作都是异步的，需通过 `synchronize_stream` 等待完成。

### 3.3 DVPP 对齐规则

DVPP 对输入输出尺寸有**对齐要求**，这是初学者最常踩的坑：

- 宽度对齐：128 字节对齐（`align_up(w, 128)`）
- 高度对齐：16 字节对齐（`align_up(h, 16)`）
- 输出格式：YUV420SP（NV12），半平面存储

> 对齐函数：`align_up(size, align) = (size + align - 1) // align * align`

该函数在 `code/yolo_postprocess.py` 中实现，供所有检测脚本调用。

## 4. AIPP 预处理加速详解

### 4.1 AIPP 卸载机制

<img src="../../images/aipp_compare.png" alt="AIPP对比" style="display: block; margin-left: 0;" />

AIPP 接管了预处理四步中的两步：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">预处理步骤</th><th style="text-align: left;">无 AIPP（CPU）</th><th style="text-align: left;">有 AIPP（NPU 硬件）</th></tr>
<tr><td style="text-align: left;">BGR→RGB</td><td style="text-align: left;">Python/OpenCV</td><td style="text-align: left;">Python/OpenCV</td></tr>
<tr><td style="text-align: left;">letterbox 缩放</td><td style="text-align: left;">Python/OpenCV</td><td style="text-align: left;">Python/OpenCV</td></tr>
<tr><td style="text-align: left;">归一化 /255</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">AIPP 硬件（NPU）</td></tr>
<tr><td style="text-align: left;">HWC→CHW 转置</td><td style="text-align: left;">Python/CPU</td><td style="text-align: left;">AIPP 硬件（NPU）</td></tr>
<tr><td style="text-align: left;">输入数据类型</td><td style="text-align: left;">float32（4 字节）</td><td style="text-align: left;">uint8（1 字节，1/4 带宽）</td></tr>
</table>

上表详细对比了有无 AIPP 时预处理各步骤的执行位置。**BGR→RGB 色彩转换**和 **letterbox 缩放**在两种路径下都由 Python/OpenCV 完成，因为这两个操作涉及非标准几何变换，AIPP 硬件不擅长处理。**归一化 /255** 和 **HWC→CHW 通道转置**是每次推理都必须做的标准化操作，AIPP 将它们从 CPU 移到 NPU 硬件，这是 AIPP 的核心价值。最关键的是**输入数据类型**的变化：无 AIPP 时需传入 float32（每像素 4 字节），有 AIPP 时只需传入 uint8（每像素 1 字节），Host→Device 传输量降为原来的 1/4，在端侧带宽受限的场景下收益显著。

### 4.2 AIPP 配置文件

实验使用的 AIPP 配置文件位于 `code/aipp_yolo.cfg`：

```protobuf
aipp_op {
  aipp_mode : static          // 静态 AIPP，转换时固化进 OM
  related_input_rank : 0      // 作用于第 0 号输入
  input_format : RGB888_U8    // 输入为 RGB uint8
  src_image_size_w : 640
  src_image_size_h : 640
  crop : false                // 不裁剪
  csc_switch : false          // 关闭色域转换（BGR→RGB 已由 Python 完成）
  // 归一化: y = (x - min_chn) * var_reci_chn，即 y = x / 255
  min_chn_0 : 0.0
  var_reci_chn_0 : 0.003921568627   // = 1/255
}
```

> **关键**：`csc_switch: false` 因为 BGR→RGB 已由 Python 完成，AIPP 不重复做。"同一件事做两遍"是 AIPP 配置最典型的错误。

## 5. YOLOv8 模型部署全流程

### 5.1 部署链路

```text
YOLOv8n (.pt) → ONNX 导出 → ATC 转换（含 AIPP）→ OM 模型 → AscendCL NPU 推理
                                       ↓
                         DVPP 硬件解码 → AIPP 硬件预处理 → AI Core 推理
```

### 5.2 ATC 转换命令

在香橙派上，ATC 转换脚本位于 `code/export_om.sh`，核心命令如下：

```bash
# 纯 OM（无 AIPP，float32 输入）
atc --model=yolov8n.onnx --framework=5 --output=../output/yolov8n_pure \
    --soc_version=Ascend310B4 --input_format=NCHW \
    --input_shape="images:1,3,640,640" --output_type=FP32

# AIPP-OM（含 AIPP，uint8 输入）
atc --model=yolov8n.onnx --framework=5 --output=../output/yolov8n_aipp \
    --soc_version=Ascend310B4 --input_format=NCHW \
    --input_shape="images:1,3,640,640" --insert_op_conf=aipp_yolo.cfg \
    --output_type=FP32
```

> **注意**：香橙派 AIPro 的 `--soc_version` 为 `Ascend310B4`，与云平台的 `Ascend910B3` 不同。

### 5.3 三条推理路径

实验对比了三条预处理路径：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">路径</th><th style="text-align: left;">预处理方式</th><th style="text-align: left;">输入类型</th><th style="text-align: left;">代码文件</th></tr>
<tr><td style="text-align: left;">Path A</td><td style="text-align: left;">OpenCV + 纯 OM（全 CPU）</td><td style="text-align: left;">float32 (4字节)</td><td style="text-align: left;">code/yolo_image_detect_opencv.py</td></tr>
<tr><td style="text-align: left;">Path B</td><td style="text-align: left;">OpenCV + AIPP-OM（CPU解码+硬件归一化）</td><td style="text-align: left;">uint8 (1字节)</td><td style="text-align: left;">—</td></tr>
<tr><td style="text-align: left;">Path C</td><td style="text-align: left;">DVPP + AIPP-OM（全 NPU 硬件）</td><td style="text-align: left;">uint8 (1字节)</td><td style="text-align: left;">code/yolo_image_detect.py</td></tr>
</table>

上表对比了三条预处理路径。**Path A（OpenCV + 纯 OM）**全部预处理在 CPU 上完成，输入为 float32（每像素 4 字节），是最慢的路径，作为基准。**Path C（DVPP + AIPP-OM）**用 NPU 硬件完成解码、缩放和归一化，输入为 uint8（每像素 1 字节），是最快的路径。实验结果通常为 Path C < Path B < Path A，DVPP+AIPP 相比纯 OpenCV 加速 3~5 倍，且 Host→Device 传输量降为 1/4。

> 性能对比脚本位于 `code/benchmark_compare.py`，在香橙派上运行可得到实际数据。

## 6. 代码目录结构

本实验的全部可执行代码存放在 `code/` 目录下，在香橙派终端中运行。检测结果输出到 `output/` 目录。

```text
Lab6_3/
├── images/                          # 测试资源目录
│   ├── dog1.jpg, dog2.jpg           # 狗的测试图片
│   ├── cat1.jpg, cat2.jpg           # 猫的测试图片
│   └── dog.mp4                      # 测试视频
├── code/                            # 可执行代码目录
│   ├── aipp_yolo.cfg                # AIPP 配置文件
│   ├── export_om.sh                 # ATC 模型转换脚本
│   ├── yolo_postprocess.py          # YOLO 后处理工具模块
│   ├── yolo_image_detect.py         # 案例一: 图片检测 (DVPP+AIPP)
│   ├── yolo_image_detect_opencv.py  # 案例一: 图片检测 (OpenCV对比)
│   ├── yolo_video_detect.py         # 案例二: 视频检测 (DVPP+AIPP)
│   ├── yolo_video_detect_opencv.py  # 案例二: 视频检测 (OpenCV对比)
│   ├── yolo_usb_camera_detect.py    # 案例三: USB摄像头实时检测+HDMI显示
│   └── benchmark_compare.py         # 三种路径性能对比基准
├── output/                          # 输出结果目录
│   ├── yolov8n_pure.om              # 纯 OM 模型 (无AIPP)
│   ├── yolov8n_aipp.om              # AIPP-OM 模型 (含AIPP)
│   ├── dvpp_aipp_dog1.png           # DVPP+AIPP 检测结果图
│   ├── opencv_dog1.png              # OpenCV 检测结果图
│   ├── dvpp_aipp_video_result.mp4   # DVPP+AIPP 视频检测结果
│   └── opencv_video_result.mp4      # OpenCV 视频检测结果
└── lab6.3_dvpp_aipp_yolo_orangepi.ipynb  # 本 Notebook
```

### 代码文件说明

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">文件</th><th style="text-align: left;">功能说明</th></tr>
<tr><td style="text-align: left;">aipp_yolo.cfg</td><td style="text-align: left;">AIPP 静态配置文件，定义 RGB888_U8 输入格式和 /255 归一化参数</td></tr>
<tr><td style="text-align: left;">export_om.sh</td><td style="text-align: left;">ATC 转换脚本，将 ONNX 转为纯 OM 和 AIPP-OM 两个模型</td></tr>
<tr><td style="text-align: left;">yolo_postprocess.py</td><td style="text-align: left;">YOLOv8 后处理工具：letterbox、输出解码、NMS、坐标还原、可视化绘制</td></tr>
<tr><td style="text-align: left;">yolo_image_detect.py</td><td style="text-align: left;">案例一：图片 YOLO 检测（DVPP 硬件解码 + AIPP 硬件归一化路径）</td></tr>
<tr><td style="text-align: left;">yolo_image_detect_opencv.py</td><td style="text-align: left;">案例一对比：图片 YOLO 检测（OpenCV 纯 CPU 预处理 + 纯 OM 推理路径）</td></tr>
<tr><td style="text-align: left;">yolo_video_detect.py</td><td style="text-align: left;">案例二：视频逐帧 YOLO 检测（DVPP + AIPP 路径）</td></tr>
<tr><td style="text-align: left;">yolo_video_detect_opencv.py</td><td style="text-align: left;">案例二对比：视频逐帧 YOLO 检测（OpenCV 纯 CPU 路径）</td></tr>
<tr><td style="text-align: left;">yolo_usb_camera_detect.py</td><td style="text-align: left;">案例三：USB 摄像头实时检测 + HDMI 显示（DVPP + AIPP 路径）</td></tr>
<tr><td style="text-align: left;">benchmark_compare.py</td><td style="text-align: left;">三种预处理路径性能对比基准测试</td></tr>
</table>

> **模型文件获取**：`yolov8n.onnx` 不再随仓库分发。请先在 `code/` 目录运行 `python3 download_yolov8n.py`（优先直连下载，失败可 `pip install ultralytics onnx` 后回退导出），再执行下方 ATC 转换。


## 7. 模型准备与 ATC 转换

### 7.1 在香橙派上的操作步骤

在香橙派终端中执行以下命令，完成模型转换：

```bash
# 1. 加载 CANN 环境
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 2. 进入 code 目录
cd code/

# 3. 执行 ATC 转换脚本（需要先准备好 yolov8n.onnx）
bash export_om.sh yolov8n.onnx ../output
```

### 7.2 转换脚本说明

`code/export_om.sh` 脚本依次执行两次 ATC 转换：

1. **纯 OM 转换**：不插入 AIPP 算子，模型输入为 float32（4.69 MB），预处理全部在 CPU 完成
2. **AIPP-OM 转换**：通过 `--insert_op_conf=aipp_yolo.cfg` 插入 AIPP 预处理算子，模型输入为 uint8（1.17 MB），归一化和通道转置由 NPU 硬件完成

转换完成后，两个 OM 模型保存在 `output/` 目录下：
- `output/yolov8n_pure.om` — 纯 OM（无 AIPP）
- `output/yolov8n_aipp.om` — AIPP-OM（含 AIPP）

> **关键差异**：AIPP-OM 的输入数据量仅为纯 OM 的 1/4（1.17 MB vs 4.69 MB），在香橙派带宽受限的端侧场景下收益显著。

在香橙派上执行 `bash export_om.sh yolov8n.onnx ../output` 的实际运行结果如下：

<img src="./images/run_results/atc_convert_result.png" alt="ATC模型转换运行结果" style="display: block; margin-left: 0;" />

---

## 8. 案例一：图片目标检测

### 8.1 案例说明

使用 `images/` 目录下的四张真实图片 `dog1.jpg`、`dog2.jpg`（狗）和 `cat1.jpg`、`cat2.jpg`（猫）进行 YOLO 目标检测。分别用 **DVPP+AIPP** 和 **OpenCV** 两种路径处理，对比性能差异。

### 8.2 DVPP + AIPP 路径（`code/yolo_image_detect.py`）

该脚本使用 DVPP 硬件解码 JPEG 图片并缩放到 640×640，然后用 AIPP-OM 模型推理（uint8 输入），最后做 YOLO 后处理并绘制检测结果。

**处理流程**：

```text
PNG图片 → OpenCV转JPEG → DVPP硬件解码(JPEG→YUV420SP) → DVPP VPC硬件缩放(→640×640)
→ YUV→RGB转换 → AIPP-OM推理(uint8输入, AIPP硬件归一化) → YOLOv8后处理(解码+NMS)
→ 坐标还原到原图 → 绘制检测框 → 保存到 output/
```

**核心代码结构**（`code/yolo_image_detect.py`）：

- `DVPPAIPPDetector` 类：封装 DVPP+AIPP 检测器
  - `_init_acl()`：初始化 AscendCL 框架
  - `_load_model()`：加载 AIPP-OM 模型
  - `_init_dvpp()`：创建 DVPP 通道
  - `_decode_jpeg()`：DVPP 硬件 JPEG 解码（JPEG → YUV420SP）
  - `_vpc_resize()`：DVPP 硬件 VPC 缩放
  - `_yuv_to_rgb()`：YUV420SP → RGB 转换（拷回 Host）
  - `detect()`：完整检测流程（解码→缩放→推理→后处理→绘制）

在香橙派上运行：

```bash
cd code/
python3 yolo_image_detect.py
```

在香橙派上运行 `python3 yolo_image_detect.py` 的实际运行结果如下：

<img src="./images/run_results/run_image_detect_dvpp_aipp.png" alt="图片检测DVPP+AIPP运行结果" style="display: block; margin-left: 0;" />

In [ ]:
# === 案例一：DVPP + AIPP 图片检测 ===
# 在香橙派终端中运行: cd code/ && python3 yolo_image_detect.py
#
# 以下为 code/yolo_image_detect.py 的核心流程说明（在香橙派上执行）

# 核心检测流程 (code/yolo_image_detect.py 中的 detect 方法):

# Step 1: 读取图片并转为 JPEG (DVPP 仅支持 JPEG 输入)
#   orig_img = cv2.imread(image_path)
#   _, jpeg_bytes = cv2.imencode('.jpg', orig_img)

# Step 2: DVPP 硬件 JPEG 解码 (JPEG → YUV420SP)
#   dev_yuv, yuv_desc, width, height = self._decode_jpeg(jpeg_data)
#   # 调用 acl.media.dvpp_jpeg_decode_async() 硬件解码

# Step 3: DVPP VPC 硬件缩放 (→ 640×640)
#   dev_resized = self._vpc_resize(yuv_desc, width, height, 640, 640)
#   # 调用 acl.media.dvpp_vpc_resize_async() 硬件缩放

# Step 4: YUV → RGB 转换 (拷回 Host, 用于 AIPP-OM 的 uint8 输入)
#   rgb_img = self._yuv_to_rgb(dev_resized, 640, 640)
#   input_data = rgb_img.transpose(2, 0, 1).reshape(1, 3, 640, 640).astype(np.uint8)

# Step 5: AIPP-OM 推理 (uint8 输入, AIPP 硬件自动做归一化 /255 和 HWC→CHW)
#   acl.rt.memcpy(input_dev, input_size, input_data, ...)  # H2D 拷贝
#   acl.mdl.execute(model_id, in_dataset, out_dataset)      # NPU 推理

# Step 6: YOLOv8 后处理 (解码 + NMS)
#   out_data = copy_output_to_host()  # D2H 拷贝
#   detections = yolov8_decode(out_data, conf_thres=0.25, iou_thres=0.45)

# Step 7: 坐标还原到原图 + 绘制检测框
#   result_img = draw_detections(orig_img, scaled_detections)
#   cv2.imwrite('output/dvpp_aipp_dog1.png', result_img)

print('案例一 DVPP+AIPP 路径: 请在香橙派上运行 code/yolo_image_detect.py')
print('预期输出: output/dvpp_aipp_dog1.png, output/dvpp_aipp_dog2.png,')
print('          output/dvpp_aipp_cat1.png, output/dvpp_aipp_cat2.png')

### 8.3 OpenCV 纯 CPU 路径（`code/yolo_image_detect_opencv.py`）

该脚本使用 OpenCV 软件解码图片并 letterbox 缩放到 640×640，然后用纯 OM 模型推理（float32 输入），用于与 DVPP+AIPP 路径对比性能。

**处理流程**：

```text
PNG图片 → OpenCV解码(BGR) → letterbox缩放(CPU) → BGR→RGB(CPU) → 归一化/255(CPU)
→ HWC→CHW转置(CPU) → float32类型转换 → 纯OM推理(float32输入) → YOLOv8后处理
→ 坐标还原到原图 → 绘制检测框 → 保存到 output/
```

在香橙派上运行：

```bash
cd code/
python3 yolo_image_detect_opencv.py
```

在香橙派上运行 `python3 yolo_image_detect_opencv.py` 的实际运行结果如下：

<img src="./images/run_results/run_image_detect_opencv.png" alt="图片检测OpenCV运行结果" style="display: block; margin-left: 0;" />

### 8.4 两种路径对比

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">对比项</th><th style="text-align: left;">DVPP + AIPP</th><th style="text-align: left;">OpenCV + 纯 OM</th></tr>
<tr><td style="text-align: left;">图片解码</td><td style="text-align: left;">DVPP 硬件 (NPU)</td><td style="text-align: left;">OpenCV 软件 (CPU)</td></tr>
<tr><td style="text-align: left;">图像缩放</td><td style="text-align: left;">DVPP VPC 硬件 (NPU)</td><td style="text-align: left;">OpenCV resize (CPU)</td></tr>
<tr><td style="text-align: left;">归一化</td><td style="text-align: left;">AIPP 硬件 (NPU)</td><td style="text-align: left;">NumPy (CPU)</td></tr>
<tr><td style="text-align: left;">输入类型</td><td style="text-align: left;">uint8 (1 字节)</td><td style="text-align: left;">float32 (4 字节)</td></tr>
<tr><td style="text-align: left;">H2D 带宽</td><td style="text-align: left;">1.17 MB</td><td style="text-align: left;">4.69 MB (4×)</td></tr>
<tr><td style="text-align: left;">单张耗时</td><td style="text-align: left;">~2-5 ms</td><td style="text-align: left;">~8-15 ms</td></tr>
<tr><td style="text-align: left;">加速比</td><td style="text-align: left;">3-5x</td><td style="text-align: left;">1x (基准)</td></tr>
</table>

---

## 9. 案例二：视频目标检测

### 9.1 案例说明

使用 `images/dog.mp4` 真实视频进行逐帧 YOLO 目标检测。视频原始分辨率 1280×720，帧率 30fps。分别用 **DVPP+AIPP** 和 **OpenCV** 两种路径逐帧处理，生成检测结果视频。

### 9.2 DVPP + AIPP 路径（`code/yolo_video_detect.py`）

该脚本使用 OpenCV `VideoCapture` 读取视频帧，逐帧用 DVPP 硬件解码+缩放和 AIPP-OM 推理，检测结果帧合成为结果视频。

**逐帧处理流程**：

```text
视频帧(BGR) → 编码JPEG → DVPP硬件解码 → DVPP VPC硬件缩放(→640×640)
→ AIPP-OM推理(uint8输入) → YOLOv8后处理 → 绘制检测框+FPS → 写入结果视频
```

**核心代码结构**（`code/yolo_video_detect.py`）：

- `DVPPAIPPVideoDetector` 类：封装视频检测器
  - `_dvpp_decode_and_resize()`：DVPP 硬件解码 + VPC 缩放
  - `detect_frame()`：单帧检测（编码→DVPP解码→DVPP缩放→AIPP推理→后处理）
- `main()` 函数：打开视频 → 逐帧检测 → 写入结果视频 → 输出统计信息

在香橙派上运行：

```bash
cd code/
python3 yolo_video_detect.py                    # 默认处理 images/dog.mp4
python3 yolo_video_detect.py /path/to/video.mp4  # 处理指定视频
```

在香橙派上运行 `python3 yolo_video_detect.py` 的实际运行结果如下：

<img src="./images/run_results/run_video_detect_dvpp_aipp_1.png" alt="视频检测DVPP+AIPP运行结果1" style="display: block; margin-left: 0;" />

<img src="./images/run_results/run_video_detect_dvpp_aipp_2.png" alt="视频检测DVPP+AIPP运行结果2" style="display: block; margin-left: 0;" />

In [ ]:
# === 案例二：DVPP + AIPP 视频检测 ===
# 在香橙派终端中运行: cd code/ && python3 yolo_video_detect.py
#
# 以下为 code/yolo_video_detect.py 的核心流程说明

# 视频逐帧检测流程:

# 1. 打开视频文件
#   cap = cv2.VideoCapture(VIDEO_PATH)
#   fps = cap.get(cv2.CAP_PROP_FPS)
#   total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 2. 创建结果视频写入器
#   writer = cv2.VideoWriter(result_path, fourcc, fps, (w, h))

# 3. 逐帧检测循环
#   while True:
#       ret, frame = cap.read()
#       if not ret: break
#       
#       # DVPP+AIPP 检测单帧
#       detections = detector.detect_frame(frame)
#       
#       # 绘制检测框 + FPS 信息
#       result_frame = draw_detections(frame, detections)
#       cv2.putText(result_frame, f'FPS: {fps:.1f}', ...)
#       
#       writer.write(result_frame)

# 4. 输出统计
#   avg_ms = total_time / frame_count
#   avg_fps = frame_count / (total_time / 1000)

print('案例二 DVPP+AIPP 路径: 请在香橙派上运行 code/yolo_video_detect.py')
print('预期输出: output/dvpp_aipp_video_result.mp4')

### 9.3 OpenCV 纯 CPU 路径（`code/yolo_video_detect_opencv.py`）

该脚本使用 OpenCV 软件预处理 + 纯 OM 推理逐帧处理视频，用于与 DVPP+AIPP 路径对比。

在香橙派上运行：

```bash
cd code/
python3 yolo_video_detect_opencv.py
```

在香橙派上运行 `python3 yolo_video_detect_opencv.py` 的实际运行结果如下：

<img src="./images/run_results/run_video_detect_opencv_1.png" alt="视频检测OpenCV运行结果1" style="display: block; margin-left: 0;" />

<img src="./images/run_results/run_video_detect_opencv_2.png" alt="视频检测OpenCV运行结果2" style="display: block; margin-left: 0;" />

### 9.4 视频检测性能对比

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">对比项</th><th style="text-align: left;">DVPP + AIPP</th><th style="text-align: left;">OpenCV + 纯 OM</th></tr>
<tr><td style="text-align: left;">逐帧预处理</td><td style="text-align: left;">DVPP 硬件 (NPU)</td><td style="text-align: left;">OpenCV (CPU)</td></tr>
<tr><td style="text-align: left;">逐帧推理</td><td style="text-align: left;">AIPP-OM (uint8)</td><td style="text-align: left;">纯 OM (float32)</td></tr>
<tr><td style="text-align: left;">平均帧率</td><td style="text-align: left;">~25-40 fps</td><td style="text-align: left;">~10-15 fps</td></tr>
<tr><td style="text-align: left;">是否实时</td><td style="text-align: left;">可达到实时 (≥25fps)</td><td style="text-align: left;">低于实时</td></tr>
</table>

> 在香橙派 310B4 上，DVPP+AIPP 路径处理 1280×720 视频通常可达 25~40 fps，满足实时检测需求；而 OpenCV 纯 CPU 路径仅 10~15 fps，无法实时。

---

## 10. 案例三：USB 摄像头实时 YOLO 检测 + HDMI 显示

### 10.1 案例说明

本案例针对昇腾香橙派开发板的特点，实现一个完整的实时目标检测系统：通过 **USB 接口** 采集摄像头视频，在 **昇腾 310B4 NPU** 上执行 YOLO 目标检测，检测结果通过 **HDMI 接口** 实时显示到连接的显示器上。

### 10.2 硬件连接

```text
┌─────────────┐    USB     ┌──────────────────────┐    HDMI    ┌──────────┐
│  USB 摄像头  │ ────────→ │  昇腾香橙派 AIPro     │ ────────→ │  显示器   │
│  (视频采集)  │            │  Ascend 310B4 NPU    │            │ (实时显示) │
└─────────────┘            │  DVPP + AIPP + YOLO  │            └──────────┘
                            └──────────────────────┘
```

硬件连接步骤：

1. 将 USB 摄像头插入香橙派的 **USB 3.0 接口**
2. 用 HDMI 线连接香橙派的 **HDMI 接口** 和显示器
3. 确认摄像头设备: `ls /dev/video*`
4. 设置摄像头权限: `sudo chmod 666 /dev/video*`

### 10.3 完整数据流

```text
USB摄像头 ──(USB接口)──→ 香橙派内存(Host)
  → OpenCV采集BGR帧
  → 编码JPEG
  → DVPP硬件解码 (JPEG → YUV420SP)         [NPU DVPP]
  → DVPP VPC硬件缩放 (→ 640×640)            [NPU DVPP]
  → AIPP-OM推理 (uint8输入, 硬件归一化)      [NPU AI Core]
  → YOLOv8后处理 (解码 + NMS)               [CPU]
  → 绘制检测框 + FPS信息                     [CPU]
  ──(HDMI接口)──→ 显示器实时显示
```

### 10.4 代码实现（`code/yolo_usb_camera_detect.py`）

**核心类结构**：

- `USBCameraYOLODetector` 类：封装 USB 摄像头实时检测器
  - `_init_acl()`：初始化 AscendCL 框架
  - `_load_model()`：加载 AIPP-OM 模型
  - `_init_dvpp()`：创建 DVPP 通道
  - `_dvpp_process()`：DVPP 硬件解码 + VPC 缩放 + YUV→RGB
  - `detect_frame()`：单帧完整检测流程
- `DRMDisplay` 类：DRM 直显模块（用于无桌面环境的 headless 模式）
- `main()` 函数：打开摄像头 → 初始化检测器 → 实时检测循环 → HDMI 显示

在香橙派上运行：

```bash
cd code/

# 方式1: OpenCV 窗口显示 (需要桌面环境, 通过 HDMI 输出到显示器)
python3 yolo_usb_camera_detect.py 0 opencv

# 方式2: DRM 直显 (无需桌面环境, 直接通过 HDMI 输出)
python3 yolo_usb_camera_detect.py 0 drm
```

**方式1：OpenCV 窗口显示**（`python3 yolo_usb_camera_detect.py 0 opencv`）的终端运行输出和 HDMI 显示器实时画面如下：

<img src="./images/run_results/run_usb_camera_opencv_terminal.png" alt="USB摄像头OpenCV模式终端运行结果" style="display: block; margin-left: 0;" />

<img src="./images/run_results/run_usb_camera_opencv_screen.png" alt="USB摄像头OpenCV模式HDMI屏幕显示" style="display: block; margin-left: 0;" />

**方式2：DRM 直显**（`python3 yolo_usb_camera_detect.py 0 drm`）的终端运行输出和 HDMI 显示器实时画面如下：

<img src="./images/run_results/run_usb_camera_drm_terminal.png" alt="USB摄像头DRM模式终端运行结果" style="display: block; margin-left: 0;" />

<img src="./images/run_results/run_usb_camera_drm_screen.png" alt="USB摄像头DRM模式HDMI屏幕显示" style="display: block; margin-left: 0;" />

运行时按键说明：
- `q`：退出
- `s`：保存当前帧截图到 `output/`
- `r`：开始/停止录像

In [ ]:
# === 案例三：USB 摄像头实时 YOLO 检测 + HDMI 显示 ===
# 在香橙派终端中运行: cd code/ && python3 yolo_usb_camera_detect.py 0 opencv
#
# 以下为 code/yolo_usb_camera_detect.py 的核心流程说明

# 实时检测主循环 (code/yolo_usb_camera_detect.py 中的 main 函数):

# Step 1: 打开 USB 摄像头
#   cap = cv2.VideoCapture(CAMERA_INDEX)  # CAMERA_INDEX=0 为第一个 USB 摄像头
#   cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
#   cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Step 2: 初始化 DVPP + AIPP 检测器
#   detector = USBCameraYOLODetector('../output/yolov8n_aipp.om')

# Step 3: 初始化 HDMI 显示
#   # 方式1: OpenCV 窗口 (通过 HDMI 输出到显示器)
#   cv2.namedWindow('YOLO Detection (Orange Pi)')
#   # 方式2: DRM 直显 (headless 模式)
#   drm_display = DRMDisplay(1280, 720)

# Step 4: 实时检测循环
#   while True:
#       ret, frame = cap.read()                    # USB 摄像头采集帧
#       detections, timing = detector.detect_frame(frame)  # DVPP+AIPP 检测
#       result_frame = draw_detections(frame, detections)  # 绘制检测框
#       
#       # 叠加 FPS 和耗时信息
#       cv2.putText(result_frame, f'FPS: {fps:.1f}', ...)
#       
#       # HDMI 显示
#       cv2.imshow('YOLO Detection', result_frame)  # OpenCV 方式
#       # 或 drm_display.show(result_frame)         # DRM 方式
#       
#       if cv2.waitKey(1) & 0xFF == ord('q'):      # 按 q 退出
#           break

print('案例三 USB摄像头实时检测: 请在香橙派上运行 code/yolo_usb_camera_detect.py')
print('硬件连接: USB摄像头 -> 香橙派USB接口, HDMI线 -> 香橙派HDMI接口 -> 显示器')
print('运行命令: python3 yolo_usb_camera_detect.py 0 opencv')

### 10.5 三种实现方式对比

在 USB 摄像头实时检测中，可以采用三种不同的预处理路径：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">方式</th><th style="text-align: left;">视频解码</th><th style="text-align: left;">图像缩放</th><th style="text-align: left;">归一化</th><th style="text-align: left;">输入类型</th><th style="text-align: left;">实时帧率</th></tr>
<tr><td style="text-align: left;">DVPP+AIPP</td><td style="text-align: left;">DVPP硬件</td><td style="text-align: left;">DVPP硬件</td><td style="text-align: left;">AIPP硬件</td><td style="text-align: left;">uint8</td><td style="text-align: left;">~30-45 fps</td></tr>
<tr><td style="text-align: left;">OpenCV+AIPP</td><td style="text-align: left;">OpenCV(CPU)</td><td style="text-align: left;">OpenCV(CPU)</td><td style="text-align: left;">AIPP硬件</td><td style="text-align: left;">uint8</td><td style="text-align: left;">~20-30 fps</td></tr>
<tr><td style="text-align: left;">纯OpenCV+纯OM</td><td style="text-align: left;">OpenCV(CPU)</td><td style="text-align: left;">OpenCV(CPU)</td><td style="text-align: left;">NumPy(CPU)</td><td style="text-align: left;">float32</td><td style="text-align: left;">~10-15 fps</td></tr>
</table>

> 本实验的 `code/yolo_usb_camera_detect.py` 采用最优的 **DVPP+AIPP** 路径，在香橙派 310B4 上可实现 30~45 fps 的实时检测帧率，满足实时应用需求。

### 10.6 HDMI 显示方式说明

香橙派 AIPro 通过 HDMI 接口连接显示器，有两种显示方式：

1. **OpenCV imshow 方式**（`opencv` 模式）：需要香橙派运行桌面环境（如 XFCE/GNOME），通过 X11 窗口系统显示。适合开发调试阶段。

2. **DRM 直显方式**（`drm` 模式）：无需桌面环境，直接通过 Linux DRM（Direct Rendering Manager）子系统操作 HDMI 显示帧缓冲区。适合嵌入式部署阶段（headless 模式）。需要安装 `pydrm` 库。

```bash
# 检查 HDMI 是否连接
xrandr --query | grep 'connected'

# 检查 DRM 设备
ls -la /dev/dri/

# 安装 pydrm (DRM 直显模式)
pip3 install pydrm
```

---

## 11. 三种路径性能对比基准

### 11.1 对比脚本说明

`code/benchmark_compare.py` 脚本对四张测试图片分别用 **Path A（OpenCV + 纯 OM）** 和 **Path C（DVPP + AIPP-OM）** 两种路径各运行 50 次取平均，对比端到端耗时和输入数据量。

在香橙派上运行：

```bash
cd code/
python3 benchmark_compare.py
```

### 11.2 预期结果

```text
=================================================================
  性能对比结果
=================================================================
  Path A (OpenCV + 纯 OM):  ~12.00 ms  (float32, 4.69 MB)
  Path C (DVPP + AIPP):     ~3.50 ms   (uint8,   1.17 MB)
  加速比: 3.4x
  带宽节省: 4.0x (float32 -> uint8)

  结论: DVPP+AIPP 将 JPEG 解码、图像缩放、归一化全部从 CPU
        移到 NPU 专用硬件，端到端加速 3.4 倍，
        且输入数据量降为 1/4，显著节省 Host->Device 带宽。
```

在香橙派上运行 `python3 benchmark_compare.py` 的实际运行结果如下：

<img src="./images/run_results/run_benchmark_compare.png" alt="性能对比基准运行结果" style="display: block; margin-left: 0;" />

> 实际数据因香橙派负载和图片内容而异，典型加速比为 3~5 倍。

## 12. 在香橙派上的完整运行流程

### 12.1 环境准备

```bash
# 1. 加载 CANN 环境
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 2. 检查 NPU 状态
npu-smi info

# 3. 检查 ATC 是否可用
which atc

# 4. 检查 AscendCL 模块
python3 -c "import acl; print('acl OK')"

# 5. 检查 OpenCV
python3 -c "import cv2; print('cv2 OK', cv2.__version__)"
```

### 12.2 模型转换

```bash
cd code/
bash export_om.sh yolov8n.onnx ../output
# 输出: output/yolov8n_pure.om, output/yolov8n_aipp.om
```

### 12.3 案例一：图片检测

```bash
cd code/

# DVPP + AIPP 路径
python3 yolo_image_detect.py
# 输出: output/dvpp_aipp_dog1.png, output/dvpp_aipp_dog2.png,
#       output/dvpp_aipp_cat1.png, output/dvpp_aipp_cat2.png

# OpenCV 对比路径
python3 yolo_image_detect_opencv.py
# 输出: output/opencv_dog1.png, output/opencv_dog2.png,
#       output/opencv_cat1.png, output/opencv_cat2.png
```

### 12.4 案例二：视频检测

```bash
cd code/

# DVPP + AIPP 路径
python3 yolo_video_detect.py
# 输出: output/dvpp_aipp_video_result.mp4

# OpenCV 对比路径
python3 yolo_video_detect_opencv.py
# 输出: output/opencv_video_result.mp4
```

### 12.5 案例三：USB 摄像头实时检测

```bash
cd code/

# 确认 USB 摄像头已连接
ls /dev/video*
sudo chmod 666 /dev/video*

# 启动实时检测 (HDMI 显示器已连接)
python3 yolo_usb_camera_detect.py 0 opencv    # OpenCV 窗口显示
# 或
python3 yolo_usb_camera_detect.py 0 drm       # DRM 直显
```

### 12.6 性能对比

```bash
cd code/
python3 benchmark_compare.py
# 输出: output/benchmark_results.txt
```

## 13. 常见问题与故障排查

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">现象</th><th style="text-align: left;">可能原因</th><th style="text-align: left;">解决方法</th></tr>
<tr><td style="text-align: left;">atc: command not found</td><td style="text-align: left;">CANN 环境未加载</td><td style="text-align: left;">source /usr/local/Ascend/ascend-toolkit/set_env.sh</td></tr>
<tr><td style="text-align: left;">DVPP 解码失败</td><td style="text-align: left;">输入图片格式不支持</td><td style="text-align: left;">转为 JPEG/YUV 格式后重试（DVPP 仅支持 JPEG）</td></tr>
<tr><td style="text-align: left;">AIPP 配置不生效</td><td style="text-align: left;">aipp_yolo.cfg 参数错误</td><td style="text-align: left;">检查 --insert_op_conf 参数与归一化配置</td></tr>
<tr><td style="text-align: left;">NPU 利用率低</td><td style="text-align: left;">预处理仍在 CPU</td><td style="text-align: left;">确认使用 AIPP-OM 而非纯 OM</td></tr>
<tr><td style="text-align: left;">DVPP 对齐错误</td><td style="text-align: left;">宽高未按 128/16 对齐</td><td style="text-align: left;">使用 align_up() 函数对齐</td></tr>
<tr><td style="text-align: left;">USB 摄像头打不开</td><td style="text-align: left;">设备未识别或权限不足</td><td style="text-align: left;">ls /dev/video*; sudo chmod 666 /dev/video*</td></tr>
<tr><td style="text-align: left;">HDMI 无显示</td><td style="text-align: left;">显示器未连接或未识别</td><td style="text-align: left;">xrandr --query 检查; 确认 HDMI 线连接</td></tr>
<tr><td style="text-align: left;">实时帧率低</td><td style="text-align: left;">未使用 DVPP+AIPP</td><td style="text-align: left;">确认使用 yolo_usb_camera_detect.py (DVPP+AIPP路径)</td></tr>
</table>

上表列出了香橙派部署中最常见的八类问题。**atc: command not found** 是最常见的环境问题，原因是 CANN 的环境变量未加载。**DVPP 解码失败**通常是因为输入图片格式不在 DVPP 支持范围内（DVPP 只支持 JPEG）。**USB 摄像头打不开**需要检查设备识别和权限。**HDMI 无显示**需要检查显示器连接和 xrandr 输出。

---

## 小结

本实验在 **昇腾香橙派 AIPro 开发板**（Ascend 310B4 NPU）上完成了 YOLOv8 目标检测的端侧部署，核心实践了：

1. **DVPP 硬件解码**：JPEG → YUV420SP → VPC缩放，全部在 NPU 专用硬件完成
2. **AIPP 预处理卸载**：归一化 /255 和 HWC→CHW 从 CPU 积到 NPU 硬件，输入降为 uint8
3. **案例一：图片检测**：对 dog1.jpg、dog2.jpg、cat1.jpg、cat2.jpg 四张图片完成 DVPP+AIPP 检测，并与 OpenCV 路径对比
4. **案例二：视频检测**：对 dog.mp4 逐帧检测，DVPP+AIPP 路径可达 25~40 fps 实时检测
5. **案例三：USB 摄像头实时检测**：USB 摄像头采集 → DVPP+AIPP 检测 → HDMI 实时显示，完整端侧实时 AI 系统
6. **三种路径对比**：DVPP+AIPP（最优）vs OpenCV+AIPP vs 纯 OpenCV，加速 3~5 倍，带宽节省 4 倍
7. **完整部署链路**：.pt → .onnx → .om（含AIPP）→ AscendCL 推理

> 通过 DVPP 与 AIPP 的协同，在香橙派 310B4 端侧设备上实现了实时 YOLO 目标检测，端到端推理耗时显著降低，输入数据传输量降为原来的 1/4。结合 USB 摄像头采集和 HDMI 显示输出，构建了完整的边缘 AI 视觉检测系统。

---

## 课后练习

**第1题**（单选题）本实验使用的目标硬件是？

- A. NVIDIA GPU
- B. 昇腾香橙派 AIPro（Ascend 310B4 NPU）
- C. gitcode CANNLab 云平台（Ascend 910B3）
- D. CPU

In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）DVPP JPEG 解码输出的图像格式是？

- A. RGB888
- B. BGR888
- C. YUV420SP（NV12）
- D. PNG

In [ ]:
q2 = ''
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）在香橙派 AIPro 上，ATC 转换时 `--soc_version` 应设为？

- A. Ascend910B3
- B. Ascend310B4
- C. Ascend910A
- D. GPU

In [ ]:
q3 = ''
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）启用 AIPP 后，输入数据从 float32 变为 uint8，带来的主要好处是？

- A. 提高计算精度
- B. 增加模型大小
- C. Host→Device 带宽节省为原来的 1/4
- D. 不需要模型转换

In [ ]:
q4 = ''
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）案例三中 USB 摄像头视频数据通过什么接口传入香橙派？

- A. HDMI
- B. USB
- C. 网线
- D. 串口

In [ ]:
q5 = ''
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）案例三中检测结果通过什么接口实时显示到显示器？

- A. USB
- B. HDMI
- C. VGA
- D. 网线

In [ ]:
q6 = ''
print(f'第{6}题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）AIPP 配置中 `csc_switch: false` 的原因是？

- A. 不需要色彩转换
- B. BGR→RGB 已由 Python 完成，AIPP 不重复做
- C. CSC 功能不可用
- D. 开启 CSC 会降低精度

In [ ]:
q7 = ''
print(f'第{7}题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）三种预处理路径中，哪个最快？

- A. OpenCV + 纯 OM（全 CPU）
- B. OpenCV + AIPP-OM（CPU 解码 + 硬件归一化）
- C. DVPP + AIPP-OM（全 NPU 硬件）
- D. 三条路径速度相同

In [ ]:
q8 = ''
print(f'第{8}题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / '06_vision_dev' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_06 import grade
grade(globals())

## 参考资料

- [昇腾香橙派 AIPro 官方文档](https://www.hiascend.com/document)
- [昇腾 DVPP 文档](https://www.hiascend.com/document)
- [昇腾 AIPP 配置指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL Python API 参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [YOLOv8 官方仓库](https://github.com/ultralytics/ultralytics)
- [CANN Toolkit 下载与安装](https://www.hiascend.com/software/cann)